# SICD to CPHD and back — chip-size sweep, with ships marked

For one Umbra ship scene this notebook:

1. downloads the scene and finds the ships
2. for each chip size (128 / 256 / 512 / 1024 / 2048) converts
   **SICD to CPHD** (inverse PFA), then refocuses the CPHD back to an image
3. shows the original chip and the reconstruction **side by side with the ship
   boxes drawn on both**, so you can see whether targets survive the round trip
4. tabulates fidelity against chip size

Why the sweep matters: the phase history of a cropped chip is missing
everything outside the crop, and the crop edges ring. Larger chips reconstruct
better. On synthetic data the round-trip correlation went 0.9968 (128 px) to
0.9999 (1024 px), so this sweep tells you the chip size your work needs.

## 1. Install and upload `sicd_to_cphd.py`

In [ ]:
!pip -q install "sarpy==2.1.1" finufft boto3 lxml shapely scipy matplotlib numpy pandas
from google.colab import files
up = files.upload()            # choose sicd_to_cphd.py
import importlib.util, sys
spec = importlib.util.spec_from_file_location("s2c", "sicd_to_cphd.py")
s2c = importlib.util.module_from_spec(spec); sys.modules["s2c"] = s2c; spec.loader.exec_module(s2c)
print("loaded sicd_to_cphd")

## 2. Find a ship scene

Umbra's ship collection is `sar-data/tasks/ship_detection_testdata/`.
If it has no SICD products, fall back to the general catalogue by changing
`PREFIX` to `sar-data/tasks/`.

In [ ]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

BUCKET = "umbra-open-data-catalog"
PREFIX = "sar-data/tasks/ship_detection_testdata/"
s3 = boto3.client("s3", region_name="us-west-2", config=Config(signature_version=UNSIGNED))

sicds, jsons = [], []
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=PREFIX):
    for o in page.get("Contents", []):
        if o["Key"].endswith("_SICD.nitf"): sicds.append((o["Size"], o["Key"]))
        elif o["Key"].endswith(".json"):    jsons.append((o["Size"], o["Key"]))
    if len(sicds) > 40: break
sicds.sort()
print(f"{len(sicds)} SICDs under {PREFIX}\n")
for i,(sz,k) in enumerate(sicds[:15]): print(f"[{i:2d}] {sz/1e6:7.1f} MB  {k.split('/')[-1]}")
if not sicds: print("none here — set PREFIX = 'sar-data/tasks/' and re-run")

In [ ]:
import os
CHOICE = 0
size, key = sicds[CHOICE]
local = "/content/" + key.rsplit("/",1)[-1]
if not os.path.exists(local):
    print(f"downloading {size/1e6:.1f} MB ..."); s3.download_file(BUCKET, key, local)
print("SICD:", local)

## 3. Open it, check the geometry, find the ships

The geometry check must pass. If it fails, this scene's conventions don't match
the script's and the conversion would be wrong — stop and report it.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from sarpy.io.complex.converter import open_complex

reader = open_complex(local)
sicd = reader.get_sicds_as_tuple()[0]
g = s2c.Geometry(sicd)                    # raises if the SICD is not PFA-formed
chk = g.check()
print("geometry check:", "PASS" if chk["ok"] else "FAIL", {k: round(v,4) for k,v in chk.items() if k!="ok"})
n_rows, n_cols = reader.get_data_size_as_tuple()[0]
print(f"image {n_rows} x {n_cols}, pixel {g.row.ss:.3f} x {g.col.ss:.3f} m, "
      f"resolution {0.886/g.row.irbw:.2f} x {0.886/g.col.irbw:.2f} m")

In [ ]:
from scipy import ndimage

def auto_boxes(img, k=6.0, min_pix=25, pad=5):
    """Bright-blob candidates. Not a detector — just finds things to look at."""
    sm = ndimage.uniform_filter(np.abs(img).astype(float), 3)
    lab, _ = ndimage.label(sm > sm.mean() + k*sm.std())
    out = []
    for i, sl in enumerate(ndimage.find_objects(lab), 1):
        if (lab[sl]==i).sum() < min_pix: continue
        out.append([max(0,sl[0].start-pad), max(0,sl[1].start-pad),
                    min(img.shape[0],sl[0].stop+pad), min(img.shape[1],sl[1].stop+pad)])
    return out

# scan a wide view (decimated) to locate ships anywhere in the scene
STEP = max(1, max(n_rows, n_cols)//2048)
wide = np.asarray(reader[::STEP, ::STEP], np.complex64)
cand = auto_boxes(wide, k=7.0)
SHIPS = [{"name": f"ship{i}", "box": [b[0]*STEP, b[1]*STEP, b[2]*STEP, b[3]*STEP]}
         for i, b in enumerate(cand)]
print(f"{len(SHIPS)} candidates in the full scene (decimation {STEP})")

# ---- or set them by hand, in FULL-image pixels:
# SHIPS = [{"name":"ship1","box":[4200,5100,4260,5190]}]

fig, ax = plt.subplots(figsize=(9,9))
a = np.abs(wide); ax.imshow(20*np.log10(a+1e-12) - 20*np.log10(a.max()), cmap="gray", vmin=-45, vmax=0)
for s in SHIPS:
    r0,c0,r1,c1 = [v/STEP for v in s["box"]]
    ax.add_patch(Rectangle((c0,r0), c1-c0, r1-r0, fill=False, ec="lime", lw=1))
ax.set_title(f"full scene (1/{STEP}) — {len(SHIPS)} candidates"); plt.show()

## 4. Pick one ship — every chip will be centred on it

Chips centred on the scene centre would lose the ship as the chip shrinks, so
the sweep centres on the target instead. That way the same ship appears in all
chip sizes and the comparison is fair.

In [ ]:
TARGET = 0                     # index into SHIPS
tb = SHIPS[TARGET]["box"]
centre = ((tb[0]+tb[2])//2, (tb[1]+tb[3])//2)
print("centre pixel:", centre, " ship box:", tb)

def chip_around(centre, size, shape):
    nr = min(size, shape[0]); nc = min(size, shape[1])
    r0 = int(np.clip(centre[0]-nr//2, 0, shape[0]-nr))
    c0 = int(np.clip(centre[1]-nc//2, 0, shape[1]-nc))
    return r0, c0, nr, nc

def boxes_in_chip(ships, chip):
    r0,c0,nr,nc = chip; out=[]
    for s in ships:
        br0,bc0,br1,bc1 = s["box"]
        br0,br1,bc0,bc1 = br0-r0, br1-r0, bc0-c0, bc1-c0
        if br1<=0 or bc1<=0 or br0>=nr or bc0>=nc: continue
        out.append((s["name"], max(0,br0), max(0,bc0), min(nr,br1), min(nc,bc1)))
    return out

## 5. The sweep: SICD to CPHD to image, at each chip size

`s2c.inverse_pfa` produces the phase history (what gets written into a CPHD
file). `refocus` is the forward direction — it images that phase history back
onto the SICD's own pixel grid, with polar density compensation and the SICD
taper re-applied so the result is directly comparable to the input.

In [ ]:
import time, pandas as pd

def refocus(sig, valid, t, f, g, origin, shape, floor=0.1):
    kr, kc = g.k_image(f, t)
    dkr, dkc = kr - g.row.kctr, kc - g.col.kctr
    dkr_dt, dkr_df = np.gradient(kr, t, f)
    dkc_dt, dkc_df = np.gradient(kc, t, f)
    jac = np.abs(dkr_df*dkc_dt - dkr_dt*dkc_df) * np.gradient(t)[:,None] * np.gradient(f)[None,:]
    v = valid
    vals = (sig[v].astype(np.complex128) * jac[v]
            * np.maximum(g.row.weight(dkr[v]), floor) * np.maximum(g.col.weight(dkc[v]), floor))
    return s2c.image_from_samples(vals, g, origin, shape, dkr[v], dkc[v])

def scores(a, b):
    a = a.ravel().astype(np.complex128); b = b.ravel().astype(np.complex128)
    corr = abs(np.vdot(a,b))/np.sqrt(np.vdot(a,a).real*np.vdot(b,b).real)
    al = np.vdot(b,a)/np.vdot(b,b).real; e = a - al*b
    return float(corr), float(10*np.log10(np.vdot(e,e).real/np.vdot(a,a).real))

CHIP_SIZES = [128, 256, 512, 1024]      # add 2048 if RAM allows
results, panels = [], []
for size in CHIP_SIZES:
    chip = chip_around(centre, size, (n_rows, n_cols))
    r0, c0, nr, nc = chip
    img = np.asarray(reader[r0:r0+nr, c0:c0+nc], np.complex64)
    t0 = time.time()
    sig, valid, t, f, fx1, fx2, has = s2c.inverse_pfa(img, g, (r0, c0))
    rec = refocus(sig, valid, t, f, g, (r0, c0), img.shape)
    dt = time.time() - t0
    corr, nmse = scores(img, rec)
    results.append({"chip": size, "pulses": sig.shape[0], "samples": sig.shape[1],
                    "CPHD MB": round(sig.nbytes/1e6, 1), "valid %": round(valid.mean()*100, 1),
                    "corr": round(corr, 4), "NMSE dB": round(nmse, 1), "sec": round(dt, 1)})
    panels.append((size, img, sig, valid, rec, boxes_in_chip(SHIPS, chip)))
    print(results[-1])
pd.DataFrame(results)

## 6. Original vs reconstruction, ships marked on both

Three columns per chip size: the SICD chip, the phase history (the CPHD
content), and the image refocused from that phase history. Green boxes are the
ships, drawn on the original **and** on the reconstruction.

In [ ]:
def db(x, floor=-45):
    v = 20*np.log10(np.abs(x)+1e-12); return np.clip(v - np.nanmax(v), floor, 0)

fig, axes = plt.subplots(len(panels), 3, figsize=(16, 5.2*len(panels)))
axes = np.atleast_2d(axes)
for row, (size, img, sig, valid, rec, bx) in enumerate(panels):
    axes[row,0].imshow(db(img), cmap="gray"); axes[row,0].set_title(f"SICD chip {size}px")
    axes[row,1].imshow(db(np.where(valid, sig, np.nan)), cmap="viridis", aspect="auto")
    axes[row,1].set_title(f"CPHD phase history {sig.shape[0]}x{sig.shape[1]}")
    axes[row,1].set_xlabel("frequency sample"); axes[row,1].set_ylabel("pulse")
    axes[row,2].imshow(db(rec), cmap="gray")
    r = [x for x in results if x["chip"]==size][0]
    axes[row,2].set_title(f"refocused from CPHD — corr {r['corr']}, {r['NMSE dB']} dB")
    for col in (0, 2):
        for name, br0, bc0, br1, bc1 in bx:
            axes[row,col].add_patch(Rectangle((bc0,br0), bc1-bc0, br1-br0, fill=False, ec="lime", lw=1.4))
            axes[row,col].text(bc0, br0-3, name, color="lime", fontsize=8)
plt.tight_layout(); plt.show()

## 7. Fidelity vs chip size, and the ships specifically

In [ ]:
df = pd.DataFrame(results)
fig, ax = plt.subplots(1, 2, figsize=(13,4.5))
ax[0].plot(df["chip"], df["corr"], "o-"); ax[0].set_xscale("log", base=2)
ax[0].set_xlabel("chip size (px)"); ax[0].set_ylabel("round-trip |corr|"); ax[0].grid(alpha=.3)
ax[1].plot(df["chip"], df["NMSE dB"], "o-"); ax[1].set_xscale("log", base=2)
ax[1].set_xlabel("chip size (px)"); ax[1].set_ylabel("NMSE (dB, lower is better)"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

# does the ship itself survive as well as the whole chip?
print(f"{'chip':>6} {'whole chip':>11} {'inside boxes':>13}")
for size, img, sig, valid, rec, bx in panels:
    if not bx: print(f"{size:>6}  (no box in chip)"); continue
    m = np.zeros(img.shape, bool)
    for _, r0, c0, r1, c1 in bx: m[r0:r1, c0:c1] = True
    print(f"{size:>6} {scores(img, rec)[0]:11.4f} {scores(img[m], rec[m])[0]:13.4f}")

## 8. Optional — write a real .cphd file at your chosen size

The sweep above keeps the phase history in memory. This writes a proper CPHD
1.x file (and runs Umbra's real CPHD comparison if one exists next to the SICD).

In [ ]:
SIZE = 1024
!python sicd_to_cphd.py "{local}" -o /content/cphd_out --chip-size {SIZE} --check --no-compare
!ls -la /content/cphd_out/*/

## Notes

* **Bigger chips reconstruct better.** The phase history of a crop is missing
  everything outside it, and the crop edge rings. On synthetic data the
  round-trip went 0.9968 at 128 px to 0.9999 at 1024 px.
* **Ships should survive at least as well as the chip average.** Bright, coherent
  scatterers occupy the centre of k-space and are the last thing to degrade. If
  the in-box correlation is *worse* than the whole chip, something is wrong —
  most likely the ship is near the chip edge.
* The chip is centred on the selected ship, not the scene centre, so the target
  stays in frame at every size.
* The auto-finder finds bright blobs, not ships. For labelled work use
  **SDFSD-v1.0**, which has rotated-box shapefiles over 211 Umbra spotlight scenes.